In [1]:
# Fetch the SRGAN code
!git clone https://github.com/Lornatang/SRGAN-PyTorch.git

# Move into the folder
%cd SRGAN-PyTorch

# Note: We are skipping the requirements.txt because Kaggle 
# already has a perfectly optimized PyTorch environment built-in!

Cloning into 'SRGAN-PyTorch'...
remote: Enumerating objects: 3631, done.
remote: Counting objects: 100% (918/918), done.
remote: Compressing objects: 100% (277/277), done.
remote: Total 3631 (delta 688), reused 814 (delta 637), pack-reused 2713 (from 1)
Receiving objects: 100% (3631/3631), 2.98 MiB | 16.84 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.
/kaggle/working/SRGAN-PyTorch


In [2]:
import os
import yaml

config_path = './configs/train/SRGAN_x4-ImageNet.yaml'

with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# 1. 90-Epoch Lock
cfg['TRAIN']['HYP']['EPOCHS'] = 90
cfg['TRAIN']['LR_SCHEDULER']['MILESTONES'] = [45, 68]

# 2. Map SAR Datasets
base_path = "/kaggle/input/datasets/maryclauds/sar-image-patches/SAR_Dataset/SAR_Dataset"
cfg['TRAIN']['DATASET']['TRAIN_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TRAIN']['DATASET']['TRAIN_LR_IMAGES_DIR'] = f"{base_path}/train_LR"
cfg['TEST']['DATASET']['PAIRED_TEST_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TEST']['DATASET']['PAIRED_TEST_LR_IMAGES_DIR'] = f"{base_path}/train_LR"

# 3. Start from Phase 1 Weights
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_G_MODEL'] = "/kaggle/input/datasets/maryclauds/sar-image-patches/epoch_90.pth.tar"
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_D_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['RESUMED_G_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['RESUMED_D_MODEL'] = ""

# 4. Fix Kaggle GPU compiled flags
cfg['MODEL']['G']['COMPILED'] = False
cfg['MODEL']['D']['COMPILED'] = False
cfg['MODEL']['EMA']['COMPILED'] = False

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

print("✅ Config written.")

✅ Config written.


In [3]:
# ── Patch utils.py cleanly (read → AST-safe string surgery → write) ──────────
utils_path = 'utils.py'

with open(utils_path, 'r') as f:
    src = f.read()

# ── FIX 1: Scheduler resume bug ───────────────────────────────────────────────
src = src.replace(
    'scheduler.load_state_dict(checkpoint["scheduler"])',
    'pass  # [HOTFIX] bypassed missing scheduler key'
)

# ── FIX 2: Rolling-checkpoint (keeps only the LATEST file) ───────────────────
# We replace the single torch.save line with a self-contained helper call,
# and inject the helper at the TOP of the file (after the last top-level import).
# This avoids any indentation / scope issue entirely.

HELPER = '''
def _rolling_save(state_dict, checkpoint_path):
    """Save checkpoint and delete the previous epoch file to save disk space."""
    import os, re
    torch.save(state_dict, checkpoint_path)
    m = re.search(r'epoch_(\\d+)', os.path.basename(checkpoint_path))
    if m:
        prev_ep = int(m.group(1)) - 1
        prev_path = re.sub(r'epoch_\\d+', f'epoch_{prev_ep}', checkpoint_path)
        if os.path.exists(prev_path):
            os.remove(prev_path)
            print(f"[Rolling] Deleted old checkpoint: {prev_path}")

'''

# Inject helper right after the last top-level `import` / `from` line
import re
last_import_match = None
for m in re.finditer(r'^(?:import |from )\S.*$', src, re.MULTILINE):
    last_import_match = m
if last_import_match:
    insert_pos = last_import_match.end()
    src = src[:insert_pos] + '\n' + HELPER + src[insert_pos:]

# Replace the bare torch.save call inside save_checkpoint with our helper
src = src.replace(
    'torch.save(state_dict, checkpoint_path)',
    '_rolling_save(state_dict, checkpoint_path)'
)

with open(utils_path, 'w') as f:
    f.write(src)

print("✅ utils.py patched cleanly — rolling backup active, scheduler bug fixed.")

✅ utils.py patched cleanly — rolling backup active, scheduler bug fixed.


In [4]:
# Verify the patch looks right before training
!grep -n "_rolling_save\|torch.save\|pass.*HOTFIX" utils.py

29:def _rolling_save(state_dict, checkpoint_path):
32:    _rolling_save(state_dict, checkpoint_path)
179:        pass  # [HOTFIX] bypassed missing scheduler key
201:    _rolling_save(state_dict, checkpoint_path)


In [5]:
!python train_gan.py --config_path ./configs/train/SRGAN_x4-ImageNet.yaml

2026-03-17 08:36:32.874763: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773736593.065188      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773736593.114808      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773736593.535627      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773736593.535683      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773736593.535687      58 computation_placer.cc:177] computation placer alr